# Unmix the Signal (ICA from scratch)
Centering → PCA whitening (covariance + `eigh`) → FastICA, all in NumPy. Run all cells top to bottom.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Audio, display
from audio_io import load_sources
from ica import ica, pca_components, center, whiten, kurtosis
from metrics import evaluate
from run import make_mixing_matrix

S, sr, names = load_sources('sources')
N = S.shape[0]
A = make_mixing_matrix(N, N, seed=0)
X = A @ S
print(names, S.shape, sr); print(np.round(A,3))

## 1. Sources and mixtures

In [ ]:
fig, ax = plt.subplots(N, 2, figsize=(12, 2*N))
for i in range(N):
    ax[i,0].plot(S[i], lw=.3); ax[i,0].set_ylabel(names[i]); ax[i,1].plot(X[i], lw=.3, c='C1')
ax[0,0].set_title('Sources'); ax[0,1].set_title('Mixtures'); plt.tight_layout(); plt.show()

## 2. Why non-Gaussianity matters (kurtosis)
Mixtures are sums of independent sources, so by the CLT they are closer to Gaussian (excess kurtosis nearer 0). ICA searches for the projections that are *least* Gaussian.

In [ ]:
print('sources  ', np.round(kurtosis(S),2))
print('mixtures ', np.round(kurtosis(X),2))

## 3. Center and whiten
Covariance C = XXᵀ/n = E D Eᵀ, Z = D^(-1/2) Eᵀ X. After this cov(Z)=I, so only a rotation is left to find.

In [ ]:
Xc,_ = center(X); Z,V,eig = whiten(Xc, N)
print('cov(Z) ≈ I:', np.allclose(np.cov(Z,bias=True), np.eye(N), atol=1e-6))
sub = slice(None,None,40)
fig, ax = plt.subplots(1,3, figsize=(13,4))
for a_,(D,t) in zip(ax,[(S,'sources'),(X,'mixtures'),(Z,'whitened')]):
    a_.scatter(D[0,sub],D[1,sub],s=2,alpha=.4); a_.set_title(t); a_.set_aspect('equal','datalim')
plt.show()

## 4. FastICA and evaluation (permutation, sign, scale handled)

In [ ]:
S_ica, W, info = ica(X, n_components=N)
S_pca = pca_components(X, N)
r_ica, est = evaluate(S, S_ica); r_pca, _ = evaluate(S, S_pca)
print('iterations', info['iterations'])
for i,n in enumerate(names):
    print(f"{n:<12s} ICA |corr|={r_ica['abs_corr'][i]:.4f} SDR={r_ica['si_sdr_db'][i]:.1f} dB | PCA |corr|={r_pca['abs_corr'][i]:.4f} SDR={r_pca['si_sdr_db'][i]:.1f} dB")

In [ ]:
for i,n in enumerate(names):
    print(n, '(original / ICA output)')
    display(Audio(S[i], rate=sr)); display(Audio(np.sign(r_ica['signed_corr'][i])*est[i]/np.abs(est[i]).max(), rate=sr))

## 5. Experiments
Run `python experiments.py` for the SNR sweep and nonlinearity comparison, then show the figure:

In [ ]:
from IPython.display import Image
Image('results/snr_sweep.png')

## 6. Discussion
Fill in your own analysis: why ICA beats PCA here, why quality falls with noise, and which nonlinearity worked best.